Multi-user Conversation Chains with ChatMessageHistory

What is Multi-user Conversation history?

Like in ChatGPT it maintain history of seperatly for each user. <user 1 history> <user 2 history>. because if keep generic history (one history from all the user), it is not correct approch. Ecah user history is different , you can not mix the history among the users. that is where we can use ChatMessageHistory.

The concept of ChatHistory refers to a class in LangChain which can be used to wrap an arbitrary chain. This ChatHistory will keep track of inputs and outputs of the underlying chain, and append them as messages to a message database. Future interactions will then load those messages and pass them into the chain as part of the input.

The beauty of ChatMessageHistory is that we can store separate conversation histories per user or session which is often the need for real-world chatbots which will be accessed by many users at the same time.

We use a get_session_history function which is expected to take in a session_id and return a Message History object. Everything is stored in memory. This session_id is used to distinguish between separate conversations, and should be passed in as part of the config when calling the new chain

In [1]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# used to retrieve conversation history from memory based on a specific user /session ID. if a given user id or session id, it will return the history.
# if user is coming fort time then make a entry in store.

In [4]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

In [10]:
history_store = {}

def get_session_history(session_id):
    if session_id not in history_store:
        history_store[session_id] = ChatMessageHistory()

    return history_store[session_id]

# prompt to load in history and current input from the user
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{human_input}"),
    ]
)
# create a basic LLM Chain
llm_chain = (prompt_template
                |
             chatgpt)

# create a conversation chain which can load memory based on specific user or session id. the advantage of RunnableWithMessageHistory constract, it
# help in creating conversation chain which load memory based on session id.
conv_chain = RunnableWithMessageHistory(
    llm_chain,
    get_session_history,
    input_messages_key="human_input",
    history_messages_key="history",
)
# create a utility function to take in current user input prompt and their session ID
# streams result live back to the user from the LLM
def chat_with_llm(prompt: str, session_id: str):
    for chunk in conv_chain.stream({"human_input": prompt},
                                   {'configurable': { 'session_id': session_id}}):
        print(chunk.content, end="")
    


In [11]:
user_id = 'bob123'
prompt = "Hi I am Bob, can you explain AI in 3 bullet points?"
chat_with_llm(prompt, user_id)

Sure, Bob! Here are three key points about AI:

1. **Definition and Purpose**: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn. Its primary purpose is to perform tasks that typically require human intelligence, such as problem-solving, understanding natural language, and recognizing patterns.

2. **Types of AI**: AI can be categorized into two main types: Narrow AI, which is designed for specific tasks (like virtual assistants or recommendation systems), and General AI, which aims to understand and reason across a wide range of tasks, similar to human cognitive abilities (though this is still largely theoretical).

3. **Applications**: AI is used in various fields, including healthcare (for diagnostics and personalized medicine), finance (for fraud detection and algorithmic trading), transportation (in autonomous vehicles), and customer service (through chatbots and virtual agents), among many others.

In [12]:

user_id = 'bob123'
prompt = "What was my name?"
chat_with_llm(prompt, user_id)

Your name is Bob. How can I assist you further, Bob?

In [13]:
user_id = 'james007'
prompt = "Hi can you explain what is an LLM in 2 bullet points?"
chat_with_llm(prompt, user_id)

Sure! Here are two key points about LLMs (Large Language Models):

- **Definition**: LLMs are advanced artificial intelligence models designed to understand and generate human-like text based on vast amounts of data. They utilize deep learning techniques, particularly transformer architectures, to process and predict language patterns.

- **Applications**: LLMs are used in various applications, including chatbots, content generation, language translation, and summarization, enabling more natural interactions between humans and machines.

In [14]:
prompt = "Actually I meant in the context of AI?"
chat_with_llm(prompt, 'james007')

Got it! Here’s a refined explanation of LLMs in the context of AI:

- **Large Language Models (LLMs)**: LLMs are a type of artificial intelligence model that is trained on extensive datasets to understand, generate, and manipulate human language. They leverage deep learning techniques, particularly transformer architectures, to capture complex language patterns and semantics.

- **Capabilities and Use Cases**: LLMs can perform a variety of language-related tasks, such as answering questions, writing essays, translating languages, and engaging in conversations, making them valuable tools in fields like customer service, content creation, and education.

In [15]:
user_id = 'bob123'
prompt = "What is the capital of portugal?"
chat_with_llm(prompt, user_id)

The capital of Portugal is Lisbon. If you have any more questions about Portugal or anything else, feel free to ask!

In [16]:
prompt = "What is the capital of Spain?"
chat_with_llm(prompt, 'james007')

The capital of Spain is Madrid.

In [17]:
chat_with_llm("what we have discussed so far", 'bob123')

So far, we have discussed the following:

1. A brief explanation of AI in three bullet points, covering its definition and purpose, types, and applications.
2. You mentioned your name, Bob.
3. You asked about the capital of Portugal, which is Lisbon.

If there's anything else you'd like to know or discuss, just let me know!

In [18]:
chat_with_llm("what we have discussed so far", 'james007')

So far, we have discussed the following topics:

1. **Large Language Models (LLMs)**: I provided a brief explanation of LLMs in the context of AI, highlighting their definition and capabilities.
2. **Capital of Spain**: You asked about the capital of Spain, and I informed you that it is Madrid. 

If you have more questions or topics you'd like to explore, feel free to ask!

Multi-user Window-based Conversation Chains with persistence - SQLChatMessageHistory

The beauty of SQLChatMessageHistory is that we can store separate conversation histories per user or session which is often the need for real-world chatbots which will be accessed by many users at the same time. Instead of in-memory we can store it in a SQL database which can be used to store a lot of conversations.

We use a get_session_history function which is expected to take in a session_id and return a Message History object. Everything is stored in a SQL database. This session_id is used to distinguish between separate conversations, and should be passed in as part of the config when calling the new chain

We also use a memory_buffer_window function to only use the top-K last historical conversations before sending it to the LLM, basically our own implementation of ConversationBufferWindowMemory

In [22]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [26]:
# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection="sqlite:///memory.db",
    )

# prompt to load in history and current input from the user
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{human_input}"),
    ]
)

# create a memory buffer window function to return the last K conversations
def memory_buffer_window(messages, k=2):
    return messages[-(k+1):]

# create a basic LLM Chain which only sends the last K conversations per user
llm_chain = (
    RunnablePassthrough.assign(history=lambda x: memory_buffer_window(x["history"]))
      |
    prompt_template
      |
    chatgpt
)
# create a conversation chain which can load memory based on specific user or session id
conv_chain = RunnableWithMessageHistory(
    llm_chain,
    get_session_history_db,
    input_messages_key="human_input",
    history_messages_key="history",
)

# create a utility function to take in current user input prompt and their session ID
# streams result live back to the user from the LLM
def chat_with_llm(prompt: str, session_id: str):
    for chunk in conv_chain.stream({"human_input": prompt},
                                   {'configurable': { 'session_id': session_id}}):
        print(chunk.content, end="")

In [27]:
user_id = 'jim001'
prompt = "Hi can you tell me which is the fastest animal?"
chat_with_llm(prompt, user_id)

The fastest animal in the world is the peregrine falcon. When in a dive, it can reach speeds of over 240 miles per hour (386 kilometers per hour). If you're considering speed in level flight, the Brazilian free-tailed bat holds the record, flying at speeds of around 99 miles per hour (160 kilometers per hour). For land animals, the cheetah is the fastest, capable of running speeds up to 60-70 miles per hour (97-113 kilometers per hour) in short bursts.

In [28]:
prompt = "what about the slowest animal?"
chat_with_llm(prompt, 'jim001')

The slowest animal is often considered to be the three-toed sloth. It moves at an average speed of about 0.24 kilometers per hour (0.15 miles per hour) when climbing trees. In water, they can swim slightly faster, but they are still quite slow compared to many other animals. Another contender for the title of slowest animal is the garden snail, which moves at a speed of about 0.03 miles per hour (0.048 kilometers per hour). Both of these animals have adapted to their slow pace in ways that suit their lifestyles.

In [29]:
user_id = 'john005'
prompt = "Explain AI in 3 bullets to a child"
chat_with_llm(prompt, user_id)

Sure! Here are three simple points to explain AI to a child:

1. **Smart Helpers**: AI is like a smart robot or computer that can help us with tasks, like answering questions or playing games, just like a friend would!

2. **Learning from Experience**: AI learns from lots of information, kind of like how you learn from your school lessons and practice. The more it learns, the better it gets at helping us.

3. **Talking and Understanding**: AI can understand what we say or write and can even talk back to us, making it feel like we’re having a conversation with a really clever buddy!

In [30]:
prompt = "what topics have we discussed, show briefly as bullet points"
chat_with_llm(prompt, 'jim001')

Sure! Here are the topics we've discussed so far:

- Fastest animal in the world (peregrine falcon)
- Speed in level flight (Brazilian free-tailed bat)
- Fastest land animal (cheetah)
- Slowest animal (three-toed sloth)
- Another contender for slowest animal (garden snail)